In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix,f1_score

In [38]:
df=pd.read_csv('shop_smart_ecommerce.csv')
df.head()


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [39]:
df.shape

(12330, 18)

In [40]:
df.isnull().sum()

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [42]:
df["VisitorType"].value_counts()

VisitorType
Returning_Visitor    10551
New_Visitor           1694
Other                   85
Name: count, dtype: int64

In [43]:
df["Revenue"].value_counts()

Revenue
False    10422
True      1908
Name: count, dtype: int64

In [44]:
y=df['Revenue'].astype('int')
X=df.drop(['Revenue'],axis=1)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [45]:
cat_columns=X.select_dtypes(include=['object']).columns
num_columns=X.select_dtypes(include=['int64','float64']).columns


#  creating a transformaation pipeline for numerical and categorical features
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
tranformer=ColumnTransformer([('num',StandardScaler(),num_columns),('cat',OneHotEncoder(handle_unknown='ignore'),cat_columns)])
pipleline=Pipeline([('preprocessor',tranformer),('model',DecisionTreeClassifier(random_state=42,class_weight="balanced"))])

In [50]:
#  finding the best parameters for the model using GridSearchCV
from sklearn.model_selection import GridSearchCV
param_grid={'model__criterion':['gini','entropy'],'model__max_depth':[2,5,10,20],'model__min_samples_split':[2,5,10,20,30],'model__min_samples_leaf':[5,10,15,20,30,50]}
grid_search=GridSearchCV(pipleline,param_grid,cv=5,scoring='f1')
grid_search.fit(X_train,y_train)
print("Best Parameters:",grid_search.best_params_)
#  training the model with the best parameters
best_model=grid_search.best_estimator_
best_model.fit(X_train,y_train)
y_pred=best_model.predict(X_test)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print("F1 Score:",f1_score(y_test,y_pred))



Best Parameters: {'model__criterion': 'gini', 'model__max_depth': 2, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}
              precision    recall  f1-score   support

           0       0.96      0.88      0.92      2055
           1       0.58      0.81      0.68       411

    accuracy                           0.87      2466
   macro avg       0.77      0.85      0.80      2466
weighted avg       0.90      0.87      0.88      2466

[[1818  237]
 [  80  331]]
F1 Score: 0.6762002042900919


In [51]:
# traing accuracy
y_train_pred=best_model.predict(X_train)
print("Training Classification Report:")
print(classification_report(y_train,y_train_pred))

Training Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.89      0.92      8367
           1       0.56      0.81      0.66      1497

    accuracy                           0.87      9864
   macro avg       0.76      0.85      0.79      9864
weighted avg       0.90      0.87      0.88      9864

